# Moduł 4: Wizualizacja danych i EDA

**EDA (Exploratory Data Analysis)** — analiza eksploracyjna: pierwsza rzecz, którą robisz z nowymi danymi.

## Spis treści
1. Matplotlib — podstawy
2. Typy wykresów
3. Seaborn — piękne wykresy statystyczne
4. Macierz korelacji i heatmapy
5. Praktyczny workflow EDA
6. Feature Engineering
7. Data Processing — obsługa danych
8. **Zadania**

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Styl
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Matplotlib — fundamenty

Matplotlib to niskopoziomowa biblioteka do tworzenia wykresów. Seaborn buduje na niej.

**Hierarchia:** `Figure` → `Axes` (subplot) → elementy (linie, punkty, tytuły)

In [ ]:
# Prosty wykres liniowy
x = np.linspace(0, 2 * np.pi, 100)
y_sin = np.sin(x)
y_cos = np.cos(x)

plt.figure(figsize=(10, 4))
plt.plot(x, y_sin, label="sin(x)", color="blue", linewidth=2)
plt.plot(x, y_cos, label="cos(x)", color="red", linestyle="--")
plt.title("Funkcje trygonometryczne")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Subploty — wiele wykresów obok siebie
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(x, y_sin)
axes[0].set_title("Sin")

axes[1].plot(x, y_cos, color="red")
axes[1].set_title("Cos")

axes[2].plot(x, y_sin ** 2 + y_cos ** 2, color="green")
axes[2].set_title("sin²+cos² = 1")

plt.tight_layout()
plt.show()

## 2. Typy wykresów — kiedy którego użyć?

| Typ | Zastosowanie | Matplotlib / Seaborn |
|-----|--------------|---------------------|
| **Liniowy** | Trendy czasowe | `plt.plot()` |
| **Punktowy (scatter)** | Relacja 2 zmiennych | `plt.scatter()` / `sns.scatterplot()` |
| **Histogram** | Rozkład jednej zmiennej | `plt.hist()` / `sns.histplot()` |
| **Słupkowy (bar)** | Porównanie kategorii | `plt.bar()` / `sns.barplot()` |
| **Boxplot** | Rozkład + outliers | `plt.boxplot()` / `sns.boxplot()` |
| **Heatmapa** | Macierz korelacji | `sns.heatmap()` |

In [ ]:
# Przygotujmy dane do wizualizacji
np.random.seed(42)
n = 200
df = pd.DataFrame({
 "wzrost": np.random.normal(170, 10, n),
 "waga": None, # wypełnimy poniżej
 "plec": np.random.choice(["K", "M"], n),
 "aktywnosc": np.random.choice(["niska", "średnia", "wysoka"], n)
})
# Waga zależna od wzrostu + szum
df["waga"] = df["wzrost"] * 0.5 - 15 + np.random.normal(0, 5, n)
df["BMI"] = df["waga"] / (df["wzrost"] / 100) ** 2
print(df.head())

In [ ]:
# SCATTER — relacja wzrost vs waga
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="wzrost", y="waga", hue="plec", alpha=0.7)
plt.title("Wzrost vs Waga")
plt.show()

In [ ]:
# HISTOGRAM — rozkład BMI
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x="BMI", bins=30, kde=True, ax=axes[0])
axes[0].set_title("Rozkład BMI")

# Histogram z podziałem na grupy
sns.histplot(data=df, x="BMI", hue="plec", bins=25, ax=axes[1])
axes[1].set_title("Rozkład BMI wg płci")

plt.tight_layout()
plt.show()

In [ ]:
# BOXPLOT — rozkład BMI per aktywność
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="aktywnosc", y="BMI", hue="plec")
plt.title("BMI wg aktywności i płci")
plt.show()
# Boxplot pokazuje: medianę (linia), Q1-Q3 (pudełko), wąsy, outliers (kropki)

## 3. Seaborn — zaawansowane wykresy

Seaborn automatycznie obsługuje kolory, legendy i formatowanie.

In [ ]:
# Pairplot — macierz ALL vs ALL (niezwykle przydatne w EDA!)
sns.pairplot(df[["wzrost", "waga", "BMI", "plec"]], hue="plec", diag_kind="kde")
plt.suptitle("Pairplot", y=1.02)
plt.show()

## 4. Macierz korelacji i heatmapy

**Korelacja Pearsona** mierzy liniowy związek między dwoma zmiennymi:
- **1.0** → idealnie dodatnia (jedna rośnie → druga rośnie)
- **0.0** → brak związku liniowego
- **-1.0** → idealnie ujemna

**UWAGA:** Korelacja ≠ przyczynowość!

In [ ]:
# Macierz korelacji
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()
print(corr_matrix)

# Heatmapa
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Macierz korelacji")
plt.show()

## 5. Praktyczny workflow EDA

Kiedy dostajesz nowy zbiór danych, zawsze wykonaj te kroki:

1. `df.head()`, `df.shape`, `df.info()` — co to za dane?
2. `df.describe()` — statystyki: min/max/średnia/odchylenie
3. `df.isna().sum()` — ile braków?
4. Histogramy każdej zmiennej numerycznej — rozkłady
5. `df.corr()` + heatmapa — jakie związki?
6. Pairplot lub scatter ważnych par
7. Boxploty — outliers
8. Analiza per kategoria (groupby + wykresy)

## 6. Feature Engineering

**Feature Engineering** to proces tworzenia nowych cech (features) z surowych danych, które lepiej reprezentują problem dla modelu ML. Jest to często **najważniejszy krok** w pipeline'ie ML.

### 6.1 One-Hot Encoding (zmienne kategoryczne)

Zmienne kategoryczne (np. kolor: "czerwony", "zielony", "niebieski") nie mają naturalnej kolejności numerycznej. **One-Hot Encoding** tworzy binarne kolumny:

| kolor | kolor_czerwony | kolor_zielony | kolor_niebieski |
|-------|:---:|:---:|:---:|
| czerwony | 1 | 0 | 0 |
| zielony | 0 | 1 | 0 |
| niebieski | 0 | 0 | 1 |

**Uwagi:**
- Dla $k$ kategorii tworzymy $k$ kolumn (lub $k-1$ z `drop_first=True` aby uniknąć multikolinearności)
- Alternatywy: **Label Encoding** (0,1,2...) — dla drzew OK, dla regresji NIE; **Target Encoding** — średnia targetu per kategoria
- `pd.get_dummies()` lub `sklearn.OneHotEncoder`

### 6.2 Feature Engineering dla szeregów czasowych (Sliding Windows)

Dla danych czasowych $(x_1, x_2, \ldots, x_T)$ tworzymy **okna przesuwne** (sliding windows):

Okno o rozmiarze $w$ generuje cechy:
- **Średnia krocząca (Moving Average):** $\text{MA}_t = \frac{1}{w}\sum_{i=0}^{w-1} x_{t-i}$
- **Odchylenie kroczące:** $\text{STD}_t = \sqrt{\frac{1}{w}\sum_{i=0}^{w-1}(x_{t-i} - \text{MA}_t)^2}$
- **Min/Max w oknie:** ekstrema lokalne
- **Lag features:** $x_{t-1}, x_{t-2}, \ldots, x_{t-k}$ (przeszłe wartości)

**Zastosowania:** prognozowanie pogody, cen akcji, anomalie w IoT, wykrywanie trendów.

### 6.3 Cechy statystyczne (Statistical Moment Features)

Z wektora surowych danych $\mathbf{x} = (x_1, \ldots, x_n)$ możemy wyliczyć **momenty statystyczne**:

| Moment | Wzór | Interpretacja |
|--------|------|---------------|
| Średnia (Mean) | $\mu = \frac{1}{n}\sum_i x_i$ | Centrum danych |
| Wariancja (Var) | $\sigma^2 = \frac{1}{n}\sum_i (x_i - \mu)^2$ | Rozrzut |
| Skośność (Skewness) | $\gamma = \frac{1}{n}\sum_i \left(\frac{x_i-\mu}{\sigma}\right)^3$ | Asymetria rozkładu |
| Kurtoza (Kurtosis) | $\kappa = \frac{1}{n}\sum_i \left(\frac{x_i-\mu}{\sigma}\right)^4 - 3$ | Ciężkość ogonów |

**Percentyle** (Q25, Q50/mediana, Q75) i **IQR** = Q75 − Q25 — odporne na outliers.

### 6.4 Operacje pooling jako cechy

**Pooling** — redukcja wymiarowości sygnalu (1D, 2D):
- **Max Pooling:** $y_i = \max(x_{i \cdot s}, \ldots, x_{i \cdot s + k - 1})$ — zachowuje najsilniejszy sygnał
- **Average Pooling:** $y_i = \frac{1}{k}\sum_{j=0}^{k-1} x_{i \cdot s + j}$ — wygładza sygnał
- **Global Average Pooling (GAP):** jeden średni wynik z całego wektora/mapy cech

Pooling jest używany zarówno w CNN (redukcja map cech) jak i do tworzenia cech z surowych danych.

### 6.5 Embeddingi jako cechy (PCA i sieci neuronowe)

Cechy mogą pochodzić z **redukcji wymiarów** lub **pre-trenowanych modeli**:

| Metoda | Typ | Opis |
|--------|-----|------|
| **PCA** | liniowa | Rzutowanie na główne składowe, zachowuje wariancję |
| **Autoenkoder** | nieliniowa | Bottleneck = embedding, odbudowuje wejście |
| **BERT/GPT** | NLP | Embeddingowy tekstu z pre-trenowanego LM |
| **ResNet** | CV | Feature extraction z przedostatniej warstwy |
| **HuBERT** | Audio | Embedding dźwięku z self-supervised modelu |

Embeddingi przekształcają dane surowe (tekst, obraz, dźwięk) w gęste wektory numeryczne, które model ML potrafi przetwarzać.

### 6.6 Feature Selection — wybór cech

Nie wszystkie cechy są przydatne. **Feature selection** = wybór podzbioru:

| Metoda | Opis |
|--------|------|
| **Filtrujące** | Korelacja z target (Pearson), mutual information, chi² — szybkie |
| **Wrappery** | Forward/Backward selection — testują podzbiory z modelem |
| **Wbudowane** | L1 (Lasso zeruje wagi), Random Forest (feature importance) |
| **Variance Threshold** | Usuwanie cech o niskiej wariancji (prawie stała) |

**Reguła kciuka:** Jeśli masz $n$ próbek, nie używaj więcej niż $\sim\sqrt{n}$ do $n/10$ cech.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt

# =========================================
# 6.1 One-Hot Encoding
# =========================================
df_cat = pd.DataFrame({"kolor": ["czerwony", "zielony", "niebieski", "czerwony", "zielony"]})
print("Oryginalne dane:")
print(df_cat)

# Pandas get_dummies
ohe = pd.get_dummies(df_cat, columns=["kolor"], prefix="kolor")
print("\nOne-Hot Encoding:")
print(ohe)

# Scikit-learn OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df_cat[["kolor"]])
print(f"\nsklearn OHE shape: {encoded.shape}")
print(f"Kategorie: {encoder.categories_[0]}")

# =========================================
# 6.2 Sliding Windows (szeregi czasowe)
# =========================================
np.random.seed(42)
t = np.arange(200)
signal = np.sin(t * 0.1) + 0.3 * np.random.randn(200) # sygnał + szum

window = 20
df_ts = pd.DataFrame({"wartość": signal})
df_ts["MA_20"] = df_ts["wartość"].rolling(window).mean()
df_ts["STD_20"] = df_ts["wartość"].rolling(window).std()
df_ts["lag_1"] = df_ts["wartość"].shift(1)
df_ts["lag_5"] = df_ts["wartość"].shift(5)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(t, signal, alpha=0.5, label="Oryginał")
axes[0].plot(t, df_ts["MA_20"], color="red", linewidth=2, label="MA(20)")
axes[0].legend()
axes[0].set_title("Średnia krocząca (Moving Average)")

axes[1].plot(t, df_ts["STD_20"], color="orange", linewidth=2)
axes[1].set_title("Odchylenie kroczące (Rolling STD)")
axes[1].set_xlabel("Czas")
plt.tight_layout()
plt.show()

# =========================================
# 6.3 Cechy statystyczne
# =========================================
from scipy.stats import skew, kurtosis

segments = [signal[i:i+50] for i in range(0, 150, 50)]
stats_features = []
for i, seg in enumerate(segments):
 stats_features.append({
 "segment": i,
 "mean": np.mean(seg),
 "std": np.std(seg),
 "skewness": skew(seg),
 "kurtosis": kurtosis(seg),
 "Q25": np.percentile(seg, 25),
 "Q75": np.percentile(seg, 75),
 "IQR": np.percentile(seg, 75) - np.percentile(seg, 25),
 })

df_stats = pd.DataFrame(stats_features)
print("\nCechy statystyczne per segment:")
print(df_stats.round(3))

# =========================================
# 6.6 Feature Selection
# =========================================
iris = load_iris()
X, y = iris.data, iris.target

# Mutual Information
mi_scores = mutual_info_classif(X, y, random_state=42)
print(f"\nMutual Information (Iris): {dict(zip(iris.feature_names, mi_scores.round(3)))}")

# Random Forest importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)
importances = rf.feature_importances_

plt.figure(figsize=(8, 4))
plt.barh(iris.feature_names, importances, color="steelblue")
plt.xlabel("Feature Importance (MDI)")
plt.title("Random Forest — ważność cech (Iris)")
plt.tight_layout()
plt.show()

# Variance Threshold
vt = VarianceThreshold(threshold=0.5)
X_filtered = vt.fit_transform(X)
kept = [name for name, keep in zip(iris.feature_names, vt.get_support()) if keep]
print(f"Cechy po Variance Threshold (>0.5): {kept} ({X_filtered.shape[1]}/{X.shape[1]})")

## 7. Data Processing — obsługa danych

Przed modelowaniem dane wymagają **preprocessingu**. Syllabus IOAI wymaga znajomości:

### 7.1 Missing Data — brakujące wartości

| Typ braków | Opis | Przykład |
|------------|------|---------|
| **MCAR** (Missing Completely at Random) | Brak niezależny od wartości | Awaria czujnika |
| **MAR** (Missing at Random) | Zależy od **innych** cech | Starsi ludzie rzadziej podają wagę |
| **MNAR** (Missing Not at Random) | Zależy od **samej** wartości | Osoby z wysokim dochodem nie podają go |

**Strategie obsługi braków:**

| Metoda | Kiedy | Wzór / Opis |
|--------|-------|-------------|
| **Usunięcie wierszy** | Mało braków (<5%), MCAR | `df.dropna()` |
| **Usunięcie kolumn** | Kolumna ma >50% braków | `df.drop(columns=[...])` |
| **Imputacja mean/median** | Dane numeryczne, MCAR/MAR | $x_{\text{fill}} = \bar{x}$ lub mediana |
| **Imputacja modą** | Dane kategoryczne | Najczęstsza wartość |
| **KNN Imputation** | Strukturalne zależności | Średnia z $k$ najbliższych sąsiadów |
| **Interpolacja** | Szeregi czasowe | Liniowa, spline, forward/backward fill |

**WAŻNE:** Imputację fitujesz na **train** i transformujesz **test** (unikaj data leakage).

### 7.2 Padding (dopełnianie)

Sekwencje o **różnych długościach** wymagają dopełnienia do jednolitego rozmiaru:

$$\text{padded}(\mathbf{x}) = [x_1, x_2, \ldots, x_T, 0, 0, \ldots, 0]_{L}$$

gdzie $L = \max_i |\mathbf{x}_i|$ w batchu.

| Typ paddingu | Opis | Użycie |
|-------------|------|--------|
| **Zero-padding** | Dopełnienie zerami | Sekwencje NLP, audio |
| **Image padding** | Obramowanie pikseli | Conv2D (`padding='same'`) |
| **Attention mask** | Maska informująca model, które pozycje są padding | Transformery (1=token, 0=padding) |

**Truncation** — obcinanie zbyt długich sekwencji do max_length.

### 7.3 Normalizacja i standaryzacja

| Metoda | Wzór | Zakres | Kiedy |
|--------|------|--------|-------|
| **Min-Max** | $x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$ | $[0, 1]$ | Gdy znany zakres |
| **Z-score (StandardScaler)** | $x' = \frac{x - \mu}{\sigma}$ | $(-\infty, +\infty)$ | Rozkład normalny |
| **Robust Scaler** | $x' = \frac{x - Q_{50}}{Q_{75} - Q_{25}}$ | $(-\infty, +\infty)$ | Outliers |
| **MaxAbs** | $x' = \frac{x}{\max|x|}$ | $[-1, 1]$ | Sparse data |
| **Layer Norm** | Normalizacja per próbkę | — | Wewnątrz sieci neuronowych |
| **Batch Norm** | Normalizacja per batch | — | Wewnątrz sieci neuronowych |

### 7.4 Augmentacja danych

Sztuczne zwiększanie zbioru treningowego:

| Typ danych | Techniki augmentacji |
|-----------|---------------------|
| **Obraz** | Flip H/V, obrót, crop, zmiana jasności/kontrastu, ColorJitter, CutOut, MixUp |
| **Tekst** | Synonym replacement, back-translation, random deletion/swap |
| **Audio** | SpecAugment (time/freq masking), pitch shift, time stretch, noise injection |
| **Tabelaryczne** | SMOTE (oversampling klasy mniejszości), noise injection |
| **Time series** | Window slicing, jittering, time warping |

**MixUp:** $\tilde{x} = \lambda x_i + (1-\lambda) x_j$, $\tilde{y} = \lambda y_i + (1-\lambda) y_j$ gdzie $\lambda \sim \text{Beta}(\alpha, \alpha)$

**CutOut:** Losowy prostokąt wypełniony zerami na obrazie.

### 7.5 Tokenizacja (powtórka z NLP)

Przekształcenie tekstu w sekwencję tokenów (indeksów):
- **Word-level:** słowa → indeksy (duży słownik)
- **BPE:** podział na częste podsłowa (GPT, Bielik)
- **WordPiece:** podobne do BPE, ale likelihood-based (BERT)
- **SentencePiece:** niezależne od języka, operuje na raw text

Szczegóły w **Moduł 11** (NLP & Embeddingi).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
import matplotlib.pyplot as plt

# =========================================
# 7.1 Missing Data — obsługa braków
# =========================================
np.random.seed(42)
df_miss = pd.DataFrame({
 "wiek": [25, 30, np.nan, 45, 50, np.nan, 35, 40],
 "dochód": [3000, np.nan, 5000, 4500, np.nan, 6000, np.nan, 5500],
 "kategoria": ["A", "B", np.nan, "A", "B", "A", np.nan, "B"]
})
print("Dane z brakami:")
print(df_miss)
print(f"\nLiczba braków na kolumnę:\n{df_miss.isna().sum()}")
print(f"% braków:\n{(df_miss.isna().mean() * 100).round(1)}")

# Imputacja mean
imp_mean = SimpleImputer(strategy="mean")
df_filled = df_miss.copy()
df_filled[["wiek", "dochód"]] = imp_mean.fit_transform(df_miss[["wiek", "dochód"]])
print(f"\nPo imputacji mean:\n{df_filled}")

# KNN Imputation
imp_knn = KNNImputer(n_neighbors=3)
df_knn = df_miss.copy()
df_knn[["wiek", "dochód"]] = imp_knn.fit_transform(df_miss[["wiek", "dochód"]])
print(f"\nPo KNN Imputation:\n{df_knn}")

# Forward fill (szeregi czasowe)
df_ffill = df_miss.copy()
df_ffill = df_ffill.ffill()
print(f"\nPo forward fill:\n{df_ffill}")

# =========================================
# 7.2 Padding
# =========================================
# Symulacja paddingu sekwencji NLP
sequences = [[1, 5, 3], [2, 7], [4, 8, 6, 1, 9]]
max_len = max(len(s) for s in sequences)

padded = np.zeros((len(sequences), max_len), dtype=int)
attention_mask = np.zeros_like(padded)

for i, seq in enumerate(sequences):
 padded[i, :len(seq)] = seq
 attention_mask[i, :len(seq)] = 1

print("Padded sequences:")
print(padded)
print("Attention mask:")
print(attention_mask)

# =========================================
# 7.3 Normalizacja — porównanie
# =========================================
np.random.seed(42)
X = np.array([[1, 100], [2, 200], [3, 300], [100, 400], [5, 500]]).astype(float)

scalers = {
 "Oryginalne": X,
 "StandardScaler": StandardScaler().fit_transform(X),
 "MinMaxScaler": MinMaxScaler().fit_transform(X),
 "RobustScaler": RobustScaler().fit_transform(X),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, data) in zip(axes, scalers.items()):
 ax.scatter(data[:, 0], data[:, 1], s=80)
 ax.set_title(name)
 ax.set_xlabel("Feature 1")
 ax.set_ylabel("Feature 2")
plt.tight_layout()
plt.show()

# =========================================
# 7.4 Augmentacja — MixUp demo
# =========================================
from sklearn.datasets import load_iris
iris = load_iris()
X_iris, y_iris = iris.data[:10], iris.target[:10]

def mixup(x1, y1, x2, y2, alpha=0.2):
 """MixUp augmentation"""
 lam = np.random.beta(alpha, alpha)
 x_mix = lam * x1 + (1 - lam) * x2
 y_mix = lam * y1 + (1 - lam) * y2
 return x_mix, y_mix

# Przykład MixUp
x_aug, y_aug = mixup(X_iris[0], y_iris[0], X_iris[5], y_iris[5])
print(f"\nMixUp example:")
print(f" x1={X_iris[0][:2]}, y1={y_iris[0]}")
print(f" x2={X_iris[5][:2]}, y2={y_iris[5]}")
print(f" x_mix={x_aug[:2].round(3)}, y_mix={y_aug:.3f}")

---
---
## Zadania do samodzielnego rozwiązania

---

### Zadanie 1: Iris — kompletna EDA (średnie)

Załaduj klasyczny zbiór Iris z seaborn (`sns.load_dataset('iris')`) i wykonaj pełną EDA:
1. Sprawdź `shape`, `info`, `describe`
2. Narysuj histogram każdej cechy (4 subploty)
3. Narysuj macierz korelacji jako heatmapę
4. Narysuj pairplot z kolorami wg gatunku
5. Boxplot każdej cechy w podziale na gatunek

In [ ]:
# Zadanie 1: Iris EDA
iris = sns.load_dataset("iris")
# TWÓJ KOD TUTAJ

### Zadanie 2: Wykres funkcji aktywacji (średnie)

Narysuj na jednym wykresie 4 funkcje aktywacji używane w sieciach neuronowych:

1. **Sigmoid:** $\sigma(x) = \frac{1}{1 + e^{-x}}$
2. **Tanh:** $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$
3. **ReLU:** $\text{ReLU}(x) = \max(0, x)$
4. **Leaky ReLU:** $\text{LeakyReLU}(x) = \max(0.01x, x)$

Zakres x: [-5, 5]. Dodaj tytuł, legendę, linie siatki.

In [ ]:
# Zadanie 2: Funkcje aktywacji
x = np.linspace(-5, 5, 200)
# TWÓJ KOD TUTAJ

### Zadanie 3: Wizualizacja rozkładów (trudne)

Wygeneruj 10000 próbek z 4 różnych rozkładów:
1. Normalny (μ=0, σ=1)
2. Jednostajny [0, 1]
3. Wykładniczy (λ=1)
4. Poisson (λ=5)

Narysuj 4 subploty (2x2) z histogramami i KDE. Dodaj tytuły ze średnią i std.

In [ ]:
# Zadanie 3: Wizualizacja rozkładów
np.random.seed(42)
# TWÓJ KOD TUTAJ

### Zadanie 4: Dashboard sprzedażowy (trudne)

Stwórz dane sprzedażowe (seed=42):
- 12 miesięcy × 4 kategorie produktów
- Losowe przychody (rosnący trend + szum)

Narysuj dashboard 2×2:
1. Wykres liniowy — przychody per kategoria w czasie
2. Słupkowy — całkowity przychód per kategoria
3. Pierścieniowy (donut) — udział % kategorii w przychodach
4. Heatmapa — przychody: miesiące × kategorie

In [ ]:
# Zadanie 4: Dashboard
np.random.seed(42)
# TWÓJ KOD TUTAJ

### Zadanie 5: Confusion matrix jako heatmapa (srednie)

1. Wygeneruj sztuczne predykcje klasyfikacji (3 klasy, 200 probek)
2. Oblicz macierz pomylek (`confusion_matrix` ze sklearn)
3. Narysuj ja jako heatmape z adnotacjami liczbowymi
4. Dodaj tytul, etykiety osi (prawdziwe vs przewidywane klasy)

In [ ]:
# Zadanie 5: Confusion matrix heatmap
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
# TWOJ KOD TUTAJ

### Zadanie 6: Wizualizacja PCA — projekcja 2D (trudne)

1. Zaladuj zbior Iris (`sklearn.datasets.load_iris`)
2. Zastosuj StandardScaler
3. Wykonaj PCA do 2 wymiarow
4. Narysuj scatter plot z kolorami per klasa
5. Dodaj strzalki ladunkow (loadings) dla oryginalnych cech
6. Ile wariancji wyjasniaja 2 pierwsze skladowe? Wypisz.

In [ ]:
# Zadanie 6: PCA wizualizacja
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
# TWOJ KOD TUTAJ